In [29]:
import numpy as np
import scipy.sparse as sp
from scipy.io import loadmat

from cqpsolver import Problem, Solver, SolverState

In [30]:
mat_dict: dict[np.ndarray] = loadmat("../QP-Test-Problems/MAT_Files/EXDATA.mat")

Q: sp.csc_array = sp.csc_array(mat_dict["Q"].astype(float))
q: np.ndarray = mat_dict["c"].astype(float)
A: sp.csc_array = sp.csc_array(mat_dict["A"].astype(float))
rl: np.ndarray = mat_dict["rl"].astype(float).flatten()
ru: np.ndarray = mat_dict["ru"].astype(float).flatten()
lb: np.ndarray = mat_dict["lb"].astype(float).flatten().reshape(-1, 1)
ub: np.ndarray = mat_dict["ub"].astype(float).flatten().reshape(-1, 1)

In [31]:
eq_mask: np.ndarray = rl == ru
A_eq: sp.csc_array = sp.csc_array(A[eq_mask])
b_eq: np.ndarray = ru[eq_mask].reshape(-1, 1)

A_eq: sp.csc_array = A_eq if A_eq.size > 0 else sp.csc_array((0, A.shape[1]))
b_eq: np.ndarray = b_eq if b_eq.size > 0 else np.zeros((0, 1))

ineq_mask: np.ndarray = np.invert(eq_mask)
G_ineq: sp.csc_array = sp.vstack([A[ineq_mask], -A[ineq_mask]], format="csc")
h_ineq: np.ndarray = np.concatenate([ru[ineq_mask], -rl[ineq_mask]]).reshape(-1, 1)

n = Q.shape[0]
G_full: sp.csc_array = sp.vstack([G_ineq, sp.eye(n), -sp.eye(n)], format="csc")
h_full: np.ndarray = np.vstack([h_ineq, ub, -lb])

finite_mask: np.ndarray = np.isfinite(h_full).flatten()
G: sp.csc_array = sp.csc_array(G_full[finite_mask])
h: np.ndarray = (h_full[finite_mask]).reshape(-1, 1)

In [32]:
prob: Problem = Problem(Q, q, G, h, A_eq, b_eq)
solver: Solver = Solver(prob, max_iter=100)
state_history: list[SolverState] = solver.solve()
final = state_history[-1]
print(f"Iter: {final.iter}, Converged: {bool(final.residuals.converged(tol=1e-8))}")
print(f"Obj: {final.obj}")
print(f"Primal ineq: {final.residuals.primal_ineq}")
print(f"Primal eq: {final.residuals.primal_eq}")
print(f"Stationarity: {final.residuals.stationarity}")
print(f"Duality: {final.residuals.duality}")

Iter: 18, Converged: True
Obj: -141.8434321889295
Primal ineq: 5.622554988882454e-14
Primal eq: 1.0552171369206694e-13
Stationarity: 1.7004206310391502e-12
Duality: 4.519322145666632e-10
